# 04 — Programación Lineal

**Curso:** Inteligencia Artificial — Optimización

Este notebook contiene ejemplos guiados y actividades para desarrollar en clase.

## Objetivos

- formular variables de decisión, función objetivo y restricciones;
- visualizar la región factible de un problema con dos variables;
- resolver un problema mediante enumeración de vértices y `scipy.optimize.linprog`;
- interpretar holguras y restricciones activas.



![Descripción](https://drive.google.com/uc?export=view&id=1lNBEfy3ZR_bHsx-LJz1L7gip_C9ZEKSb)

## 1. Problema de producción

Una empresa produce dos artículos, $A$ y $B$.

| Recurso | Producto A | Producto B | Disponible |
|---|---:|---:|---:|
| Máquina | 2 h | 1 h | 100 h |
| Trabajo | 1 h | 3 h | 90 h |

La utilidad es de 40 unidades por cada $A$ y 50 por cada $B$.

Variables de decisión:

- $x_1$: unidades del producto $A$;
- $x_2$: unidades del producto $B$.

Modelo:

$$
\max 40x_1+50x_2
$$

sujeto a

$$
2x_1+x_2\leq100
$$

$$
x_1+3x_2\leq90
$$

$$
x_1,x_2\geq0
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog

## 2. Visualización de la región factible

In [ ]:
x1 = np.linspace(0, 60, 500)
constraint_1 = 100 - 2 * x1
constraint_2 = (90 - x1) / 3
upper = np.minimum(constraint_1, constraint_2)
upper = np.maximum(upper, 0)

plt.figure(figsize=(8, 5))
plt.plot(x1, constraint_1, label=r'$2x_1+x_2=100$')
plt.plot(x1, constraint_2, label=r'$x_1+3x_2=90$')
plt.fill_between(x1, 0, upper, where=(upper >= 0), alpha=0.25, label='Región factible')
plt.xlim(0, 60)
plt.ylim(0, 40)
plt.xlabel(r'$x_1$')
plt.ylabel(r'$x_2$')
plt.title('Región factible')
plt.legend()
plt.grid(True)
plt.show()

## 3. Solución mediante vértices

En un problema lineal con una región factible acotada, si existe un óptimo, al menos un vértice es óptimo. Para dos variables podemos calcularlos explícitamente.

In [ ]:
# Intersecciones con los ejes y entre las dos restricciones.
A = np.array([[2, 1], [1, 3]], dtype=float)
b = np.array([100, 90], dtype=float)
intersection = np.linalg.solve(A, b)

vertices = [
    np.array([0.0, 0.0]),
    np.array([50.0, 0.0]),
    np.array([0.0, 30.0]),
    intersection
]


def profit(x):
    return 40 * x[0] + 50 * x[1]

for vertex in vertices:
    print(f'x={vertex}, utilidad={profit(vertex):.2f}')

best_vertex = max(vertices, key=profit)
print()
print('Mejor vértice:', best_vertex)
print('Utilidad máxima:', profit(best_vertex))

## 4. Solución con SciPy

`linprog` minimiza. Para maximizar $c^Tx$, minimizamos $-c^Tx$.

In [ ]:
c = np.array([-40, -50])
A_ub = np.array([[2, 1], [1, 3]])
b_ub = np.array([100, 90])
bounds = [(0, None), (0, None)]

result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

if not result.success:
    raise RuntimeError(result.message)

print('Solución:', result.x)
print('Utilidad máxima:', -result.fun)
print('Holguras:', result.ineqlin.residual)

Una holgura igual a cero indica que la restricción está **activa** en la solución. En este problema, ambos recursos se usan completamente.

In [ ]:
solution = result.x

plt.figure(figsize=(8, 5))
plt.plot(x1, constraint_1, label=r'$2x_1+x_2=100$')
plt.plot(x1, constraint_2, label=r'$x_1+3x_2=90$')
plt.fill_between(x1, 0, upper, where=(upper >= 0), alpha=0.25, label='Región factible')
plt.scatter([solution[0]], [solution[1]], s=130, marker='P', label='Óptimo')
plt.xlim(0, 60)
plt.ylim(0, 40)
plt.xlabel(r'$x_1$')
plt.ylabel(r'$x_2$')
plt.title('Solución óptima')
plt.legend()
plt.grid(True)
plt.show()

## 5. Análisis de sensibilidad básico

Modificaremos la utilidad del producto $B$ y observaremos cuándo cambia la solución óptima.

In [ ]:
profits_b = np.arange(10, 101, 5)
solutions = []

for p_b in profits_b:
    r = linprog([-40, -p_b], A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    solutions.append((p_b, r.x[0], r.x[1], -r.fun))

print('utilidad_B | x1 | x2 | utilidad total')
for p_b, x_1, x_2, value in solutions:
    print(f'{p_b:>10.0f} | {x_1:>5.1f} | {x_2:>5.1f} | {value:>14.1f}')

## Actividades

### Actividad 1 — Restricción adicional

Agrega una restricción de almacenamiento:

$$
4x_1+2x_2\leq160
$$

Grafica la nueva región factible y resuelve el modelo.

In [ ]:
# TODO: agrega la tercera restricción.
A_ub_extended = None
b_ub_extended = None

# result_extended = linprog(...)

### Actividad 2 — Producción mínima

El contrato exige producir al menos 10 unidades de $B$:

$$
x_2\geq10
$$

Representa esta condición correctamente en `linprog` y resuelve el problema.

### Actividad 3 — Inviabilidad

Crea una restricción adicional que vuelva el problema inviable. Comprueba el estado retornado por SciPy y explica qué significa.

### Actividad 4 — Variables enteras

¿Tiene sentido producir 37.5 unidades de un artículo? Investiga la diferencia conceptual entre programación lineal y programación lineal entera. No es necesario implementar un solver entero en este notebook.

## Preguntas de cierre

1. ¿Qué diferencia hay entre una restricción y la función objetivo?
2. ¿Qué significa que una solución sea factible?
3. ¿Por qué `linprog` recibe el negativo de las utilidades?
4. ¿Toda solución óptima usa por completo todos los recursos?